# Phân bố lớp: DiffMul không SupCon vs DiffMul + SupCon thuần (M1/M2)
Chỉ đổi đúng 1 yếu tố = SupCon. Ảnh 4 panel + bảng số liệu.


In [ ]:
import base64, io, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_pure'
CLEAN.mkdir(parents=True, exist_ok=True)
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK/'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(p.stdout.readline, ''):
            print(line, end='', flush=True); lf.write(line); lf.flush()
    p.stdout.close()
    rc = p.wait()
    if rc != 0: raise RuntimeError(f'rc={rc}: {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar: tar.extractall(repo_root)
os.chdir(repo_root)
print(os.getcwd(), flush=True)


In [ ]:
import torch
print('CUDA', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available(): raise SystemError('No GPU')
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow matplotlib')


In [ ]:
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
ckpt_dir = repo_root / 'results/ckpts'
ckpt_dir.mkdir(parents=True, exist_ok=True)
for f in Path('/kaggle/input').rglob('*.pt'):
    dest = ckpt_dir / f.name
    if not dest.exists(): shutil.copy2(f, dest)
cands = list(Path('/kaggle/input').rglob('bcb_test.parquet'))
if cands and not (data_raw/'bcb_test.parquet').exists(): shutil.copy2(cands[0], data_raw/'bcb_test.parquet')
print('ckpts', len(list(ckpt_dir.glob('*.pt'))), 'test', (data_raw/'bcb_test.parquet').exists(), flush=True)


In [ ]:
t0 = time.perf_counter()
sh('python scripts/analyze_supcon_pure.py --batch-size 32 --pairs-file data/raw/bigclonebench/bcb_test.parquet --out-dir results/supcon_pure')
print(f'XONG {(time.perf_counter()-t0)/60:.1f} phut', flush=True)
print((repo_root/'results/supcon_pure/report.json').read_text(encoding='utf-8'), flush=True)


In [ ]:
out = repo_root/'results/supcon_pure'
for p in out.glob('*'):
    if p.is_file() and p.suffix in ('.json','.png'):
        name = 'pure_' + p.name
        shutil.copy2(p, CLEAN/name); shutil.copy2(p, WORK/name)
        print('SAVED', name, flush=True)
os.chdir(WORK)
for junk in ('project','data','clean_pure'): shutil.rmtree(WORK/junk, ignore_errors=True)
print('DONE', flush=True)
